# Imports and Initialization

In [ ]:
import qlib
from qlib.config import REG_US # [REG_CN, REG_US]

from qlib.utils import init_instance_by_config
from qlib.utils import flatten_dict
from qlib.workflow import R


import pandas as pd
import numpy as np

import torch
from torch.backends import cuda

from pathlib import Path
import sys
import warnings


p = str(Path().resolve()).split("QuantitativeML_Investing")[0] + "QuantitativeML_Investing"


# Add custom library to path
append_path = p + "\.Lib"
sys.path.append(append_path)


# Init Qlib
provider_uri = p + "\Database\qlib_data_US"  # target_dir
qlib.init(provider_uri=provider_uri, region=REG_US)
print("Qlib version: " + str(qlib.__version__))


print(p)


# GPU
print("GPU enabled: " + str(torch.cuda.is_available()) + " at device: " + str(torch.cuda.current_device()))


# Warnings
warnings.filterwarnings("ignore")

In [ ]:
instruments = 'all'
benchmark = '^GSPC'

# Data and Model

## Data Loader

In [ ]:
###################################
# Define DataLoader
###################################
from qlib.data.dataset.loader import QlibDataLoader
from furlag_qlib import qdatasets, qfilters  # Custom Class


##################################################################################################################
# --------------
# Features
# --------------

fields, names = qdatasets.dataset_features_ts_50_v1()

##################################################################################################################
# --------------
# Labels
# --------------

labels = ['''((EMA(Ref($close, -5), 5) / EMA($close, 5)) - 1)''']
# labels = ['''((Mean(Ref($close, -5), 5) / Mean($close, 5)) - 1)''']

label_names = ['label']

##################################################################################################################
# --------------
# Filters
# --------------

filters = qfilters.Filters()

##################################################################################################################
##################################################################################################################
# ----------------------------
# Initialize data loader
# ----------------------------

data_loader = QlibDataLoader(config={"feature": (fields, names), "label": (labels, label_names)},
                             filter_pipe=filters.us, freq="day")

print(data_loader)
print("Number of features == " + str(len(fields)))

## Data Handler

In [ ]:
###################################
# Define DataHandler
###################################
from qlib.data.dataset.handler import DataHandler, DataHandlerLP
from qlib.data.dataset.processor import ProcessInf, RobustZScoreNorm, Fillna, DropnaProcessor, DropnaLabel, CSZScoreNorm

from furlag_qlib.qprocessors import DropLabelOutliners  # Custom Class


defined_datahandler = DataHandlerLP(instruments=instruments,
                                    # start_time='2000-01-01',
                                    # end_time='2022-01-01',
                                    data_loader=data_loader,
                                    infer_processors=[
                                            # ProcessInf(),
                                            RobustZScoreNorm(
                                                fields_group="feature",
                                                fit_start_time="2000-01-01", fit_end_time="2019-12-31",
                                                clip_outlier="False"
                                            ),
                                            Fillna(fields_group="feature", fill_value=0),
                                            # DropnaProcessor(fields_group="feature")
                                       ],
                                    learn_processors=[
                                            DropnaLabel(),
                                            # DropLabelOutliners(0.2),
                                            CSZScoreNorm(fields_group="label")
                                       ]
                                    )

## Task Definition

In [ ]:
###################################
# Define task
###################################

task = {
    "model": {
            "class": "ALSTM",
            "module_path": "furlag_qlib.model.pytorch_alstm_ts",
            "kwargs": {
                'd_feat': len(fields),
                'early_stop': 10,
                'n_epochs': 200,
                
                'batch_size': 2048,  # 2048
                'hidden_size': 32,  # 128
                'num_layers': 2,  # 2
                'dropout': 0.0,  # 0
                'lr': 0.001,  # 0.0001
                'rnn_type': 'GRU',  # 'GRU', 'LSTM'
                
                'metric': 'loss',
                'loss': 'mse',

                'seed': 42,

                'n_jobs': 0,
                'GPU': 0
            },
    },
    "dataset": {
        "class": "TSDatasetH",
        "module_path": "qlib.data.dataset",
        "kwargs": {
            "step_len": 10,
            "handler": defined_datahandler,
            "segments": {
                "train": ("2000-01-01", "2019-12-31"),
                "valid": ("2020-01-01", "2023-12-31"),
                "test": ("2020-01-01", "2023-12-31"),
            },
        },
    },
}


###################################
# Initialize dataset
###################################

dataset = init_instance_by_config(task["dataset"])

print(dataset)


###################################
# Initialize model
###################################

model = init_instance_by_config(task["model"])

print(model)

## Model Training/Loading

In [ ]:
###################################
# Model
###################################

train = True

if train:
    # Train model
    with R.start(experiment_name='train', recorder_name='train_recorder'):
        R.log_params(**flatten_dict(task))

        # model.fit(dataset)
        model.train_n_epochs(dataset, n_epochs=20)
        
        R.save_objects(trained_model=model)
        rid = R.get_recorder().id

else:
    # Load model
    recorder = R.get_recorder(experiment_name='train', recorder_name='train_recorder')
    model = recorder.load_object("trained_model")

# Backtest

## Port Analysis Config

In [ ]:
###################################
# Define port analysis config
###################################
from furlag_qlib import strategies  # Custom Class
from furlag_qlib.stop_loss import StopLossLogics  # Custom Class


# StopLoss logics (for Strategy)
sll = StopLossLogics()
atr_sll = sll.atr_stoploss(diff=1.5, of='$close', r=14)


# ------------------------------------------------------------


port_analysis_config = {
    "strategy": {
        "class": "BasicStrategy",
        "module_path": "furlag_qlib.strategies",
        "kwargs": {
            "signal": (model, dataset),
            
        },
    },

    "backtest": {
        "account": 10000,
        "benchmark": benchmark,

        "start_time": None,
        "end_time": None,

        "exchange_kwargs": {
                "freq": "day",
                "deal_price": "open",
                "open_cost": 0.0005,
                "close_cost": 0.0005,
                "min_cost": 2,
        },
    },

    "executor": {
        "class": "SimulatorExecutor",
        "module_path": "qlib.backtest.executor",
        "kwargs": {
            "time_per_step": "day",
            "generate_portfolio_metrics": True,
        },
    }

}

## Loop Backtest

In [ ]:
###################################
# Loop Testing
###################################

from qlib.workflow.record_temp import SignalRecord, PortAnaRecord
from IPython.display import clear_output


info_df = pd.DataFrame()


# backtest and analysis
with R.start(experiment_name='test', recorder_name='test_recorder'):
    recorder = R.get_recorder()
    ba_rid = recorder.id

    for i in range(0, 20):
        print("##########################################################################################################################################")
        print("Epoch " + str(i) + ":")
        model.load_model_at_epoch(i)


        # prediction
        sr = SignalRecord(model, dataset, recorder)
        sr.generate()

        # backtest & analysis
        par = PortAnaRecord(recorder, port_analysis_config, "day")
        par.generate()


        analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")


        info_df = info_df.append(analysis_df.transpose())



clear_output(wait=True)



# Display
info_df.reset_index(drop=True, inplace=True)

info_df.to_csv("results.csv")

with pd.option_context('display.max_rows', 100, 'display.max_columns', 100):  # more options can be specified
    display(info_df)

## Prediction & Analysis

In [ ]:
###################################
# Load model at n epoch of training
###################################

model.load_model_at_epoch(50)

In [ ]:
###################################
# prediction, backtest & analysis
###################################
from qlib.workflow.record_temp import SignalRecord, PortAnaRecord


with R.start(experiment_name='test', recorder_name='test_recorder'):
    # recorder = R.get_recorder(recorder_id=rid, experiment_name="train")
    # model = recorder.load_object("trained_model")

    # prediction
    recorder = R.get_recorder()
    ba_rid = recorder.id
    sr = SignalRecord(model, dataset, recorder)
    sr.generate()

    # backtest & analysis
    par = PortAnaRecord(recorder, port_analysis_config, "day")
    par.generate()

In [ ]:
from qlib.contrib.report import analysis_model, analysis_position
from qlib.data import D

recorder = R.get_recorder(recorder_id=ba_rid, experiment_name='test')


pred_df = recorder.load_object("pred.pkl")

pred_df_dates = pred_df.index.get_level_values(level='datetime')


report_normal_df = recorder.load_object("portfolio_analysis/report_normal_1day.pkl")
positions = recorder.load_object("portfolio_analysis/positions_normal_1day.pkl")
analysis_df = recorder.load_object("portfolio_analysis/port_analysis_1day.pkl")

## Analysis Position

### Report

In [ ]:
analysis_position.report_graph(report_normal_df)

### Risk Analysis

In [ ]:
analysis_position.risk_analysis_graph(analysis_df, report_normal_df)

### Save

In [ ]:
report_normal_df.to_csv("Analysis/report_normal.csv")
analysis_df.to_csv("Analysis/port_analysis.csv")

In [ ]:
pred_df.to_csv("Analysis/predictions.csv")

In [ ]:
# Save positions
positions_df = pd.DataFrame.from_dict(positions, orient='index')
positions_df.to_csv("Analysis/positions.csv")